# Production-like agent prompt and cost smoke test

This notebook builds a prompt shaped like production without submitting a competition prediction. It attempts a read-only lookup of a previous Explaining Markets event summary, falls back to labeled sample bullets if the API does not expose a usable historical summary URL, samples a random valid dossier and industry playbook, calls the configured model, reports token/cost metadata, and strictly validates the JSON response.

The sampled summary, industry, and dossier intentionally do not need to refer to the same company; this is a format and cost smoke test.

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Literal
import json
import os
import random
import re
import sys

import httpx
import yaml
from dotenv import load_dotenv
from litellm import completion, completion_cost, token_counter
from pydantic import BaseModel, ConfigDict, Field

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / '.env', override=False)

from predict import MODEL, LLM_MAX_RETRIES, LLM_TIMEOUT_SECONDS, _required_api_key
from prompt_construction import (
    DOSSIER_PATH, INDUSTRY_PATH, PROMPT_PATH,
    is_valid_dossier, load_prompt_rules, load_yaml,
)

API_BASE_URL = os.getenv('EM_API_BASE_URL', 'https://api.explainingmarkets.ai/v1').rstrip('/')
API_KEY = os.getenv('EM_API_KEY')
RANDOM_SEED = 7  # Change or set to None for a different sample.
rng = random.Random(RANDOM_SEED)
print('Model:', MODEL)
print('Explaining Markets API key loaded:', bool(API_KEY))
print('Provider key loaded:', bool(os.getenv(_required_api_key(MODEL))))


## 1. Try to load a previous event summary

The API request is read-only. Historical calendar responses do not always contain a still-valid information URL, so failure cleanly selects the fallback summary. No prediction is posted.

In [ ]:
FALLBACK_SUMMARY = (
    '- Quarterly revenue exceeded consensus by approximately 4%.\n'
    '- Adjusted EPS beat consensus, helped by gross-margin expansion.\n'
    '- Management raised full-year revenue guidance but maintained EPS guidance.\n'
    '- Demand remained healthy while management noted higher second-half input costs.'
)


def event_list(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ('events', 'items', 'data'):
            if isinstance(payload.get(key), list):
                return payload[key]
    return []


def summary_text(payload):
    if isinstance(payload, str) and payload.strip():
        return payload.strip()
    if not isinstance(payload, dict):
        return None
    for key in ('summary', 'event_summary', 'bullets', 'facts'):
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, list) and value:
            return '\n'.join(f'- {item}' for item in value)
    return None


def fetch_previous_summary(days_back: int = 120):
    if not API_KEY:
        return FALLBACK_SUMMARY, {'source': 'fallback', 'reason': 'EM_API_KEY is not set'}
    yesterday = datetime.now(timezone.utc).date() - timedelta(days=1)
    start = yesterday - timedelta(days=days_back)
    try:
        response = httpx.get(
            f'{API_BASE_URL}/events',
            params={'start_date': start.isoformat(), 'end_date': yesterday.isoformat()},
            headers={'X-API-Key': API_KEY}, timeout=30.0,
        )
        response.raise_for_status()
        events = event_list(response.json())
        rng.shuffle(events)
        for event in events:
            inline = summary_text(event)
            if inline:
                return inline, {'source': 'api_inline', 'event_id': event.get('event_id') or event.get('id')}
            url = next((event.get(key) for key in ('information_url', 'summary_url') if event.get(key)), None)
            if not url:
                continue
            try:
                detail = httpx.get(url, timeout=15.0)
                detail.raise_for_status()
                text = summary_text(detail.json())
                if text:
                    return text, {'source': 'api_information_url', 'event_id': event.get('event_id') or event.get('id')}
            except (httpx.HTTPError, ValueError):
                continue
        return FALLBACK_SUMMARY, {'source': 'fallback', 'reason': 'No usable historical summary in API response'}
    except (httpx.HTTPError, ValueError) as exc:
        return FALLBACK_SUMMARY, {'source': 'fallback', 'reason': f'{type(exc).__name__}: {exc}'}


event_summary, summary_metadata = fetch_previous_summary()
print('Summary metadata:', summary_metadata)
print(event_summary[:1200])


## 2. Sample a valid dossier and random industry

A dossier is valid only when `reaction_statistics.observations > 0`, matching production prompt construction.

In [ ]:
valid_dossiers = []
for path in sorted(DOSSIER_PATH.glob('*.yaml')):
    try:
        candidate = load_yaml(path)
    except (OSError, yaml.YAMLError):
        continue
    if is_valid_dossier(candidate):
        valid_dossiers.append((path, candidate))

if not valid_dossiers:
    raise RuntimeError(f'No positive-observation dossiers found in {DOSSIER_PATH}')
dossier_path, dossier = rng.choice(valid_dossiers)

industry_playbooks = load_yaml(INDUSTRY_PATH)
industry_candidates = [
    key for key, value in industry_playbooks.items()
    if key != 'quarter_calibration' and isinstance(value, dict) and value.get('rules')
]
if not industry_candidates:
    raise RuntimeError(f'No industry playbooks found in {INDUSTRY_PATH}')
industry = rng.choice(industry_candidates)
ticker = str(dossier.get('ticker') or dossier_path.stem).upper()

print('Sampled dossier:', dossier_path.name)
print('Dossier observations:', dossier['reaction_statistics']['observations'])
print('Random industry:', industry)


## 3. Construct the production-shaped prompt

This uses the same prompt template and `load_prompt_rules` function as production. The only deliberate test override is pairing a random industry with the sampled dossier.

In [ ]:
def construct_test_prompt(summary: str, industry: str, dossier: dict) -> str:
    template = PROMPT_PATH.read_text(encoding='utf-8')
    template = re.sub(r'\A\s*<!--.*?-->\s*', '', template, count=1, flags=re.DOTALL)
    rules = load_prompt_rules(industry, include_dossier_rule=True)
    return (
        template
        .replace('{event_bullets}', summary)
        .replace('{core_directive}', rules.core_directive)
        .replace('{precedence}', rules.precedence)
        .replace('{anti_patterns}', rules.anti_patterns)
        .replace('{global_rules}', rules.global_rules)
        .replace('{industry_rules}', rules.industry_rules)
        .replace('{dossier}', yaml.safe_dump(dossier, sort_keys=False))
    )


user_prompt = construct_test_prompt(event_summary, industry, dossier)
messages = [{'role': 'system', 'content': user_prompt}]
try:
    estimated_input_tokens = token_counter(model=MODEL, messages=messages)
except Exception:
    estimated_input_tokens = None
print(f'Prompt characters: {len(user_prompt):,}')
print('Estimated input tokens:', estimated_input_tokens)
print('Unresolved placeholders:', sorted(set(re.findall(r'\{[a-z_]+\}', user_prompt))))
assert not re.findall(r'\{(?:event_bullets|core_directive|industry_rules|dossier)\}', user_prompt)


## 4. Call the model and validate strict JSON

This is the only cell that incurs model cost. It fails early if the provider key selected by `MODEL` is missing.

In [ ]:
class AgentPrediction(BaseModel):
    model_config = ConfigDict(extra='forbid')
    expected_abnormal_return_pct: float
    predicted_percentile: float = Field(ge=0, le=1)
    direction: Literal['up', 'neutral', 'down']
    confidence: Literal['high', 'medium', 'low']
    top_drivers: list[str]
    rules_applied: list[str]

provider_key_name = _required_api_key(MODEL)
if not os.getenv(provider_key_name):
    raise RuntimeError(f'{provider_key_name} is not set; add it to .env before running the paid model cell')

response = completion(
    model=MODEL, messages=messages, response_format={'type': 'json_object'},
    temperature=0, timeout=LLM_TIMEOUT_SECONDS, num_retries=LLM_MAX_RETRIES,
)
raw_content = response.choices[0].message.content
prediction = AgentPrediction.model_validate_json(raw_content)

usage = getattr(response, 'usage', None)
usage_data = usage.model_dump() if hasattr(usage, 'model_dump') else dict(usage or {})
try:
    cost_usd = completion_cost(completion_response=response)
except Exception:
    cost_usd = getattr(response, '_hidden_params', {}).get('response_cost')

print('Validated JSON response:')
print(prediction.model_dump_json(indent=2))
print('Usage:', json.dumps(usage_data, indent=2, default=str))
print('Estimated response cost (USD):', cost_usd if cost_usd is not None else 'unavailable for this provider/model')
prediction
